In [0]:
%python
# 1. Exact path confirmed from your Databricks screenshot
folder_path = "/Volumes/workspace/default/nyc_taxi_project/nyc_taxi_project/"

# 2. Extract: Read all 3 yellow taxi parquet files into one table
df_trips = spark.read.parquet(folder_path + "*.parquet")

# 3. Extract: Read the CSV lookup file
df_zones = spark.read.csv(folder_path + "taxi_zone_lookup.csv", header=True, inferSchema=True)

# 4. Data Quality Check: Print starting row counts
print("Total raw taxi trips:", df_trips.count())
print("Total taxi zones:", df_zones.count())

# 5. Preview the first 5 rows so we can verify exact column names
display(df_trips.limit(5))


Total raw taxi trips: 11198026
Total taxi zones: 265


VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1,2025-03-01T00:17:16.000,2025-03-01T00:25:52.000,1,0.9,1,N,140,236,1,7.9,3.5,0.5,2.6,0.0,1.0,15.5,2.5,0.0,0.0
1,2025-03-01T00:37:38.000,2025-03-01T00:43:51.000,1,0.6,1,N,140,262,1,6.5,3.5,0.5,2.3,0.0,1.0,13.8,2.5,0.0,0.0
2,2025-03-01T00:24:35.000,2025-03-01T00:39:49.000,1,1.94,1,N,161,68,1,14.9,1.0,0.5,5.16,0.0,1.0,25.81,2.5,0.0,0.75
2,2025-03-01T00:56:16.000,2025-03-01T01:01:35.000,2,0.95,1,N,231,13,1,7.2,1.0,0.5,2.59,0.0,1.0,15.54,2.5,0.0,0.75
1,2025-03-01T00:01:44.000,2025-03-01T00:10:00.000,1,1.5,1,N,163,236,1,8.6,4.25,0.5,2.85,0.0,1.0,17.2,2.5,0.0,0.75


In [0]:
%python
from pyspark.sql.functions import col, hour, month, round

# 1. Data Quality: Keep only valid taxi trips
df_clean = df_trips.filter(
    (col("trip_distance") > 0) &
    (col("fare_amount") > 0) &
    (col("passenger_count") > 0) &
    (col("tpep_pickup_datetime").isNotNull()) &
    (col("tpep_dropoff_datetime").isNotNull())
)

# 2. Transform: Add helpful columns for analytics and partitioning
df_transformed = df_clean \
    .withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_month", month(col("tpep_pickup_datetime"))) \
    .withColumn("tip_percentage", round((col("tip_amount") / col("fare_amount")) * 100, 2))

# 3. Join Pickup Zone names from df_zones
pu_zones = df_zones.select(
    col("LocationID").alias("PU_ID"),
    col("Borough").alias("PU_Borough"),
    col("Zone").alias("PU_Zone")
)
df_with_pu = df_transformed.join(pu_zones, df_transformed["PULocationID"] == pu_zones["PU_ID"], "left")

# 4. Join Dropoff Zone names from df_zones
do_zones = df_zones.select(
    col("LocationID").alias("DO_ID"),
    col("Borough").alias("DO_Borough"),
    col("Zone").alias("DO_Zone")
)
df_final = df_with_pu.join(do_zones, df_with_pu["DOLocationID"] == do_zones["DO_ID"], "left")

# 5. Check how many clean rows remain after removing bad data
print("Clean valid taxi trips:", df_final.count())
display(df_final.select("tpep_pickup_datetime", "pickup_hour", "PU_Zone", "DO_Zone", "fare_amount", "tip_percentage").limit(5))


Clean valid taxi trips: 8579020


tpep_pickup_datetime,pickup_hour,PU_Zone,DO_Zone,fare_amount,tip_percentage
2025-03-01T00:17:16.000,0,Lenox Hill East,Upper East Side North,7.9,32.91
2025-03-01T00:37:38.000,0,Lenox Hill East,Yorkville East,6.5,35.38
2025-03-01T00:24:35.000,0,Midtown Center,East Chelsea,14.9,34.63
2025-03-01T00:56:16.000,0,TriBeCa/Civic Center,Battery Park City,7.2,35.97
2025-03-01T00:01:44.000,0,Midtown North,Upper East Side North,8.6,33.14


In [0]:
%python
# Print the exact list of all column names in df_trips
print(df_trips.columns)


['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']


In [0]:
%python
# 1. Load: Save the clean DataFrame as a permanent, partitioned Delta Table
df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("pickup_month") \
    .saveAsTable("workspace.default.nyc_taxi_clean")

print("Success! Clean partitioned Delta table saved as: workspace.default.nyc_taxi_clean")


Success! Clean partitioned Delta table saved as: workspace.default.nyc_taxi_clean


In [0]:
%sql
SELECT 
    pickup_hour,
    COUNT(*) AS total_trips,
    ROUND(AVG(fare_amount), 2) AS avg_fare
FROM workspace.default.nyc_taxi_clean
GROUP BY pickup_hour
ORDER BY pickup_hour ASC;


pickup_hour,total_trips,avg_fare
0,214760,19.22
1,141965,17.16
2,91911,16.14
3,61961,17.0
4,40554,22.91
5,46458,27.51
6,105332,22.07
7,221073,18.62
8,316592,17.19
9,369316,17.3


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT 
    CASE 
        WHEN payment_type = 1 THEN 'Credit Card'
        WHEN payment_type = 2 THEN 'Cash'
        WHEN payment_type = 3 THEN 'No Charge'
        WHEN payment_type = 4 THEN 'Dispute'
        ELSE 'Other'
    END AS payment_method,
    COUNT(*) AS total_trips,
    ROUND(AVG(tip_percentage), 2) AS avg_tip_percent
FROM workspace.default.nyc_taxi_clean
GROUP BY payment_type
ORDER BY total_trips DESC;


payment_method,total_trips,avg_tip_percent
Credit Card,7369802,26.14
Cash,1056915,0.0
Dispute,115231,0.06
No Charge,37072,0.14


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT 
    PU_Zone AS pickup_neighborhood,
    DO_Zone AS dropoff_neighborhood,
    COUNT(*) AS trip_count,
    ROUND(AVG(trip_distance), 2) AS avg_miles,
    ROUND(AVG(total_amount), 2) AS avg_total_fare
FROM workspace.default.nyc_taxi_clean
WHERE PU_Zone IS NOT NULL AND DO_Zone IS NOT NULL
GROUP BY PU_Zone, DO_Zone
ORDER BY trip_count DESC
LIMIT 10;


pickup_neighborhood,dropoff_neighborhood,trip_count,avg_miles,avg_total_fare
Upper East Side South,Upper East Side North,67336,1.04,15.52
Upper East Side North,Upper East Side South,57337,1.02,15.8
Upper East Side North,Upper East Side North,45846,0.62,13.14
Upper East Side South,Upper East Side South,44356,0.64,13.83
Midtown Center,Upper East Side South,31350,1.05,17.94
Upper East Side South,Midtown Center,27248,1.04,17.87
Midtown Center,Upper East Side North,26619,1.92,22.64
Upper West Side South,Upper West Side North,25461,0.83,13.76
Lincoln Square East,Upper West Side South,24837,0.98,15.0
Upper West Side South,Lincoln Square East,23498,0.86,14.36
